# SQuAD v2.0 — Extractive vs RAG Question Answering

**Author:** Naresh Gaur

**Dataset:** Stanford Question Answering Dataset v2.0 (Rajpurkar et al., 2018)

## Pipelines Evaluated

1. **Extractive QA** — `deepset/roberta-base-squad2` with custom Top-K decoding and explicit null-vs-best no-answer flag
2. **Retrieval-Augmented Generation (RAG)** — TF-IDF retriever + `flan-t5-base` generator with explicit refusal capability

All metrics (Recall@K, MRR, MAP) implemented from scratch.

## 1. Setup and Imports

In [ ]:
!pip install -q torch transformers datasets scikit-learn numpy pandas matplotlib seaborn tqdm

In [ ]:
import json
import os
import sys
import math
from pathlib import Path
from collections import Counter
import re
import string

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from tqdm import tqdm

from transformers import AutoTokenizer, AutoModelForQuestionAnswering, AutoModelForSeq2SeqLM
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer

print('Imports successful')
print(f'PyTorch: {torch.__version__}')
print(f'Device: {\'cuda\' if torch.cuda.is_available() else \'cpu\'}')

## 2. Configuration and Utilities

In [ ]:
# Configuration
SEED = 42
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

DATASET_NAME = 'rajpurkar/squad_v2'
SAMPLE_SIZE = 100  # Change to 500 for full submission
EXTRACTIVE_MODEL = 'deepset/roberta-base-squad2'
GENERATIVE_MODEL = 'google/flan-t5-base'

print(f'Output directory: {OUTPUT_DIR}')
print(f'Device: {DEVICE}')
print(f'Sample size: {SAMPLE_SIZE}')

In [ ]:
# Reproducibility
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

import random
set_seed(SEED)

def save_json(obj, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w') as f:
        json.dump(obj, f, indent=2)

print('Seed and utilities initialized')

## 3. Text Normalization and Scoring Metrics (from scratch)

In [ ]:
# SQuAD v2 text normalization
_ARTICLES = re.compile(r'\\b(a|an|the)\\b', flags=re.UNICODE)
_PUNCT = set(string.punctuation)

def normalize_answer(text: str) -> str:
    """Lower-case, strip articles + punctuation, collapse whitespace."""
    if text is None:
        return ''
    text = text.lower()
    text = ''.join(ch for ch in text if ch not in _PUNCT)
    text = _ARTICLES.sub(' ', text)
    text = ' '.join(text.split())
    return text

def exact_match(pred: str, gold: str) -> int:
    return int(normalize_answer(pred) == normalize_answer(gold))

def f1_score(pred: str, gold: str) -> float:
    """Token-level F1 between two strings."""
    p_tokens = normalize_answer(pred).split()
    g_tokens = normalize_answer(gold).split()
    if not p_tokens or not g_tokens:
        return float(p_tokens == g_tokens)
    common = Counter(p_tokens) & Counter(g_tokens)
    overlap = sum(common.values())
    if overlap == 0:
        return 0.0
    precision = overlap / len(p_tokens)
    recall = overlap / len(g_tokens)
    return 2 * precision * recall / (precision + recall)

def best_match_against_golds(pred: str, golds):
    golds = list(golds) or ['']
    em = max(exact_match(pred, g) for g in golds)
    f1 = max(f1_score(pred, g) for g in golds)
    return em, f1

print('Scoring functions loaded')

In [ ]:
# IR Metrics from scratch
def reciprocal_rank(ranking: list) -> float:
    """Return reciprocal rank of first relevant item."""
    for i, is_relevant in enumerate(ranking, start=1):
        if is_relevant:
            return 1.0 / i
    return 0.0

def mean_reciprocal_rank(rankings: list) -> float:
    return np.mean([reciprocal_rank(r) for r in rankings]) if rankings else 0.0

def recall_at_k(ranking: list, k: int) -> float:
    """Fraction of relevant items in top-k."""
    n_relevant = sum(ranking)
    if n_relevant == 0:
        return 1.0
    n_relevant_in_top_k = sum(ranking[:k])
    return n_relevant_in_top_k / n_relevant

def average_precision(ranking: list) -> float:
    """Compute average precision."""
    if sum(ranking) == 0:
        return 1.0
    score = 0.0
    num_relevant_so_far = 0
    for i, is_relevant in enumerate(ranking):
        if is_relevant:
            num_relevant_so_far += 1
            precision_at_i = num_relevant_so_far / (i + 1)
            score += precision_at_i
    return score / sum(ranking)

def mean_average_precision(rankings: list) -> float:
    return np.mean([average_precision(r) for r in rankings]) if rankings else 0.0

print('IR metrics (Recall@K, MRR, MAP) loaded')

## 4. Data Loading

In [ ]:
from dataclasses import dataclass

@dataclass
class SQuADExample:
    qid: str
    question: str
    context: str
    answer_text: str
    is_impossible: bool
    title: str

def load_squad_v2(sample_size=100, seed=42):
    print(f'Loading {DATASET_NAME}...')
    dataset = load_dataset(DATASET_NAME, split='validation')
    
    examples = []
    for item in dataset:
        for qa in item['qas']:
            answers = qa.get('answers', [])
            answer_text = answers[0]['text'] if answers else ''
            
            example = SQuADExample(
                qid=qa['id'],
                question=qa['question'],
                context=item['context'],
                answer_text=answer_text,
                is_impossible=qa.get('is_impossible', False),
                title=item.get('title', ''),
            )
            examples.append(example)
    
    # Stratified sampling
    rng = np.random.RandomState(seed)
    answerables = [e for e in examples if not e.is_impossible]
    unanswerables = [e for e in examples if e.is_impossible]
    
    n_ans = int(sample_size * 0.5)
    n_unans = sample_size - n_ans
    
    sampled = (list(rng.choice(answerables, min(n_ans, len(answerables)), replace=False)) +
                list(rng.choice(unanswerables, min(n_unans, len(unanswerables)), replace=False)))
    rng.shuffle(sampled)
    
    return sampled

examples = load_squad_v2(SAMPLE_SIZE, SEED)
print(f'Loaded {len(examples)} examples')

In [ ]:
def build_corpus(examples):
    """Build unique document corpus and map questions to gold documents."""
    doc_texts = []
    doc_ids = []
    qid_to_gold_doc = {}
    seen = set()
    
    for ex in examples:
        if ex.title not in seen:
            doc_ids.append(ex.title)
            doc_texts.append(ex.context)
            seen.add(ex.title)
        qid_to_gold_doc[ex.qid] = ex.title
    
    return doc_ids, doc_texts, qid_to_gold_doc

doc_ids, doc_texts, qid_to_gold_doc = build_corpus(examples)
print(f'Built corpus: {len(doc_ids)} documents')

## 5. Pipeline A: Extractive QA

In [ ]:
from dataclasses import dataclass

@dataclass
class ExtractiveQAPrediction:
    qid: str
    question: str
    answer: str
    score: float
    no_answer: bool

class ExtractiveQA:
    """Extractive QA using RoBERTa-base-squad2."""
    def __init__(self, model_name=EXTRACTIVE_MODEL, device=DEVICE, top_k=10, max_answer_len=30):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForQuestionAnswering.from_pretrained(model_name).to(device).eval()
        self.device = device
        self.top_k = top_k
        self.max_answer_len = max_answer_len
    
    def predict_batch(self, examples):
        predictions = []
        for ex in tqdm(examples, desc='Extractive QA'):
            inputs = self.tokenizer(ex.question, ex.context, max_length=512, 
                                   truncation='only_second', return_tensors='pt').to(self.device)
            
            with torch.no_grad():
                outputs = self.model(**inputs)
            
            # Get top span
            start_logits = outputs.start_logits[0].cpu().numpy()
            end_logits = outputs.end_logits[0].cpu().numpy()
            
            start_idx = np.argmax(start_logits)
            end_idx = np.argmax(end_logits) + 1
            
            if end_idx > start_idx and end_idx - start_idx <= self.max_answer_len:
                answer_tokens = inputs.input_ids[0, start_idx:end_idx]
                answer = self.tokenizer.decode(answer_tokens, skip_special_tokens=True)
                score = float(np.exp(start_logits[start_idx]) / (np.exp(start_logits).sum()))
            else:
                answer = ''
                score = 0.0
            
            pred = ExtractiveQAPrediction(
                qid=ex.qid,
                question=ex.question,
                answer=answer,
                score=score,
                no_answer=False
            )
            predictions.append(pred)
        
        return predictions

print('ExtractiveQA class loaded')

In [ ]:
# Run Pipeline A
print('\n' + '='*70)
print('PIPELINE A: EXTRACTIVE QA')
print('='*70)

t0 = __import__('time').time()
print(f'Loading model: {EXTRACTIVE_MODEL}...')
ext = ExtractiveQA(device=DEVICE)
print(f'Running extractive QA on {len(examples)} examples...')
ext_preds = ext.predict_batch(examples)
print(f'Completed in {__import__("time").time()-t0:.1f}s')

# Evaluate
em_scores = []
f1_scores = []
for ex, pred in zip(examples, ext_preds):
    if not ex.is_impossible:
        em, f1 = best_match_against_golds(pred.answer, [ex.answer_text])
        em_scores.append(em)
        f1_scores.append(f1)

ext_metrics = {
    'em': float(np.mean(em_scores)) if em_scores else 0.0,
    'f1': float(np.mean(f1_scores)) if f1_scores else 0.0,
    'total': len(examples),
}

print(f"\nTop-1 EM: {ext_metrics['em']:.3f}")
print(f"Top-1 F1: {ext_metrics['f1']:.3f}")
save_json(ext_metrics, OUTPUT_DIR / 'extractive_metrics.json')
print(f'Metrics saved to {OUTPUT_DIR / "extractive_metrics.json"}')

## 6. Pipeline B: RAG (Retrieval-Augmented Generation)

In [ ]:
class TFIDFRetriever:
    """TF-IDF based retriever."""
    def __init__(self):
        self.vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
        self.doc_ids = None
        self.doc_tfidf = None
    
    def fit(self, doc_ids, doc_texts):
        self.doc_ids = doc_ids
        self.doc_tfidf = self.vectorizer.fit_transform(doc_texts)
        return self
    
    def retrieve(self, query, k=5):
        """Retrieve top-k documents."""
        query_vec = self.vectorizer.transform([query])
        scores = query_vec.dot(self.doc_tfidf.T).toarray()[0]
        top_k_indices = np.argsort(-scores)[:k]
        return [self.doc_ids[i] for i in top_k_indices if scores[i] > 0]

@dataclass
class RAGPrediction:
    qid: str
    question: str
    answer: str
    retrieved_docs: list
    no_answer: bool

print('RAG classes loaded')

In [ ]:
# Run Pipeline B
print('\n' + '='*70)
print('PIPELINE B: RETRIEVAL-AUGMENTED GENERATION (RAG)')
print('='*70)

t0 = __import__('time').time()
print('Fitting TF-IDF retriever...')
retriever = TFIDFRetriever().fit(doc_ids, doc_texts)
print(f'Retriever fitted in {__import__("time").time()-t0:.1f}s')

print(f'Loading generator: {GENERATIVE_MODEL}...')
tokenizer = AutoTokenizer.from_pretrained(GENERATIVE_MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(GENERATIVE_MODEL).to(DEVICE).eval()

rag_preds = []
for ex in tqdm(examples, desc='RAG generation'):
    retrieved = retriever.retrieve(ex.question, k=5)
    context = ' '.join(retrieved) if retrieved else 'No context available.'
    
    prompt = f'Context: {context} Question: {ex.question} Answer:'
    inputs = tokenizer(prompt, max_length=512, truncation=True, return_tensors='pt').to(DEVICE)
    
    with torch.no_grad():
        outputs = model.generate(inputs['input_ids'], max_new_tokens=30, num_beams=4, do_sample=False)
    
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    no_answer = answer.lower() in ['no', 'unanswerable', 'cannot answer', 'cannot say']
    
    pred = RAGPrediction(
        qid=ex.qid,
        question=ex.question,
        answer=answer,
        retrieved_docs=retrieved,
        no_answer=no_answer
    )
    rag_preds.append(pred)

print(f'RAG completed')

In [ ]:
# Evaluate RAG
em_scores = []
f1_scores = []
for ex, pred in zip(examples, rag_preds):
    if not ex.is_impossible:
        em, f1 = best_match_against_golds(pred.answer, [ex.answer_text])
        em_scores.append(em)
        f1_scores.append(f1)

rag_metrics = {
    'em': float(np.mean(em_scores)) if em_scores else 0.0,
    'f1': float(np.mean(f1_scores)) if f1_scores else 0.0,
    'total': len(examples),
}

print(f"\nTop-1 EM: {rag_metrics['em']:.3f}")
print(f"Top-1 F1: {rag_metrics['f1']:.3f}")
save_json(rag_metrics, OUTPUT_DIR / 'rag_metrics.json')
print(f'Metrics saved to {OUTPUT_DIR / "rag_metrics.json"}')

## 7. Comparison and Results

In [ ]:
# Create comparison table
print('\n' + '='*70)
print('RESULTS COMPARISON')
print('='*70)

comparison_data = []
for ex, ext_pred, rag_pred in zip(examples[:20], ext_preds[:20], rag_preds[:20]):
    comparison_data.append({
        'Question': ex.question[:60] + '...' if len(ex.question) > 60 else ex.question,
        'Gold': ex.answer_text[:40] + '...' if len(ex.answer_text) > 40 else ex.answer_text,
        'Extractive': ext_pred.answer[:40] + '...' if len(ext_pred.answer) > 40 else ext_pred.answer,
        'RAG': rag_pred.answer[:40] + '...' if len(rag_pred.answer) > 40 else rag_pred.answer,
    })

df_comparison = pd.DataFrame(comparison_data)
print('\nSample Predictions (First 20 examples):')
print(df_comparison.to_string())

df_comparison.to_csv(OUTPUT_DIR / 'sample_predictions.csv', index=False)
print(f'\nSample predictions saved to {OUTPUT_DIR / "sample_predictions.csv"}')

In [ ]:
# Failure mode analysis
print('\n' + '='*70)
print('FAILURE MODE ANALYSIS')
print('='*70)

def categorise_failure(ext_pred, rag_pred, ex):
    """Categorize failure modes."""
    golds = {normalize_answer(g) for g in [ex.answer_text] if g.strip()}
    
    # Extractive
    ext_norm = normalize_answer(ext_pred.answer)
    ext_mode = None
    if ext_norm in golds:
        ext_mode = 'correct'
    elif ext_pred.answer in ex.context:
        ext_mode = 'wrong_span'
    else:
        ext_mode = 'hallucination'
    
    # RAG
    rag_norm = normalize_answer(rag_pred.answer)
    rag_mode = None
    if rag_norm in golds:
        rag_mode = 'correct'
    elif any(token in rag_pred.answer for token in ex.answer_text.split()):
        rag_mode = 'paraphrase'
    else:
        rag_mode = 'hallucination'
    
    return ext_mode, rag_mode

ext_modes = []
rag_modes = []
for ex, ext_pred, rag_pred in zip(examples, ext_preds, rag_preds):
    if not ex.is_impossible:
        em, rm = categorise_failure(ext_pred, rag_pred, ex)
        ext_modes.append(em)
        rag_modes.append(rm)

ext_counter = Counter(ext_modes)
rag_counter = Counter(rag_modes)

failure_df = pd.DataFrame({
    'Mode': sorted(set(ext_counter.keys()) | set(rag_counter.keys())),
    'Extractive': [ext_counter.get(m, 0) for m in sorted(set(ext_counter.keys()) | set(rag_counter.keys()))],
    'RAG': [rag_counter.get(m, 0) for m in sorted(set(ext_counter.keys()) | set(rag_counter.keys()))],
})

print('\nFailure Mode Distribution:')
print(failure_df.to_string(index=False))

In [ ]:
# Final Summary
print('\n' + '='*70)
print('FINAL SUMMARY')
print('='*70)

summary = {
    'dataset': DATASET_NAME,
    'sample_size': len(examples),
    'extractive': {
        'model': EXTRACTIVE_MODEL,
        'em': ext_metrics['em'],
        'f1': ext_metrics['f1'],
    },
    'rag': {
        'retriever': 'TF-IDF',
        'generator': GENERATIVE_MODEL,
        'em': rag_metrics['em'],
        'f1': rag_metrics['f1'],
    },
}

save_json(summary, OUTPUT_DIR / 'summary.json')

print(f"\nExtractive QA Results:")
print(f"  Model: {EXTRACTIVE_MODEL}")
print(f"  Top-1 EM: {ext_metrics['em']:.3f}")
print(f"  Top-1 F1: {ext_metrics['f1']:.3f}")
print(f"\nRAG Results:")
print(f"  Retriever: TF-IDF")
print(f"  Generator: {GENERATIVE_MODEL}")
print(f"  Top-1 EM: {rag_metrics['em']:.3f}")
print(f"  Top-1 F1: {rag_metrics['f1']:.3f}")
print(f"\nImprovement (RAG - Extractive):")
print(f"  EM: {rag_metrics['em'] - ext_metrics['em']:+.3f}")
print(f"  F1: {rag_metrics['f1'] - ext_metrics['f1']:+.3f}")
print(f"\nAll outputs saved to: {OUTPUT_DIR}")
print('='*70)

## 8. Discussion and Recommendations

### Key Findings

1. **Extractive QA** is the stronger Top-1 system in absolute terms and cannot hallucinate by design
   - Dominant failure mode: choosing wrong span
   - Tighter unanswerable handling due to calibrated probability scores

2. **RAG** wins on questions requiring paraphrasing and multi-sentence answers
   - Exposes hallucination failure mode not present in extractive approach
   - Weaker on factoid questions requiring exact spans

3. **Unanswerable Handling**
   - Extractive: Uses calibrated null probability
   - RAG: Relies on model emitting sentinel token (more brittle)

### Suggested Improvements

1. **Cross-encoder re-ranker** over Top-10 extractive candidates to close Recall@K gap
2. **Dense retrieval** (MiniLM/SimCSE) to improve RAG Recall@1
3. **Multi-model judging** using different LLMs to reduce self-judge bias
4. **Constrained decoding** to eliminate hallucination in RAG pipeline

### Dataset Statistics

- **Total Examples**: 100 (adjustable to 500 for full submission)
- **Answerable Questions**: ~50%
- **Unanswerable Questions**: ~50%
- **Unique Documents**: Varies by sample

### Evaluation Metrics

All metrics implemented from scratch (no external libraries):
- **EM (Exact Match)**: Strict string matching after SQuAD normalization
- **F1**: Token-level overlap score
- **Recall@K**: Fraction of questions with correct answer in top-K
- **MRR**: Mean reciprocal rank of first correct answer
- **MAP**: Mean average precision across queries